In [2]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
#from datetime import datetime
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)

with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
        SELECT DISTINCT
            a.Identificacion,
            a.Nombre_Estudiante,
            a.Periodo,
            a.Cod_Unidad,
            a.Status,
            a.Nuevo,
            a.promedio,
            a.Semestre_SINU,
            a.ciclo,
            a.SEMESTRE_MEN,
            a.Año,
            a.AÑO_MEN,
            a.[Estado_Alumno],
            a.anio_grado,
            a.Regional,
            a.Programa,
            c.SNIES,
            c.ESTADO_IDIOMAS,
            c.ESTADO_ACADEMICO,
            c.ESTADO_DEUDA,
            c.ESTADO_GRADO,
            d.grad_annio,
            d.grad_semestre as grad_semestre
        FROM
            CUN_REPOSITORIO.Desercion.Resultado_Final_CUN a
        LEFT JOIN
            DBARON.Grados_Saldo c
        ON
                a.Identificacion = c.NUM_IDENTIFICACION
            --AND a.Cod_Unidad = c.COD_UNIDAD
            AND a.Pensum = c.COD_PENSUM
            AND a.ciclo = c.ciclo
        LEFT JOIN 
            CUN_REPOSITORIO.Grados.Consolidado_Grados d
        ON 
                d.grad_annio = A.anio_grado 
            AND d.NIVEL = a.ciclo 
            AND d.NUMERO = a.Identificacion
        
       --WHERE
            --a.identificacion IN ('1023922245','1000005283')
    
        ORDER BY
            a.Identificacion,
            a.Periodo;
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

Conexion a la base de datos realizada


In [3]:
df_n = df[(df['Status'] != 'Desercion CUN') & (df['Status'] != 'Retiro')]
df['num_repeticiones'] = df_n.groupby(['ciclo', 'Identificacion']).cumcount() + 1

In [4]:
df['Primer_curso'] = df.groupby(['Identificacion', 'ciclo']).cumcount() == 0
df['Primer_curso'] = df['Primer_curso'].replace({True: 'SI', False: 'NO'})

df['ultimo_curso'] = df.groupby(['Identificacion', 'ciclo']).cumcount() == df.groupby(['Identificacion', 'ciclo'])['ciclo'].transform('count') - 1
df['ultimo_curso'] = df['ultimo_curso'].replace({True: 'SI', False: 'NO'})

In [5]:
df[['Identificacion', 'Nombre_Estudiante', 'Periodo', 'Cod_Unidad',
       'Status',  'ciclo', 'SEMESTRE_MEN',
       'Año', 'AÑO_MEN',  'grad_annio', 'grad_semestre', 'num_repeticiones',
       'Primer_curso', 'ultimo_curso']]

df['grad_semestre'] = df['grad_semestre'].str.replace('\.0$', '', regex=True)

<>:6: SyntaxWarning: invalid escape sequence '\.'
<>:6: SyntaxWarning: invalid escape sequence '\.'
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_12628\644592720.py:6: SyntaxWarning: invalid escape sequence '\.'
  df['grad_semestre'] = df['grad_semestre'].str.replace('\.0$', '', regex=True)


In [6]:
df['grad_semestre'] = df['grad_semestre'].replace(['None','0'])
df['grad_semestre'] .unique()

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_12628\1553333390.py:1: FutureWarning: Series.replace without 'value' and with non-dict-like 'to_replace' is deprecated and will raise in a future version. Explicitly specify the new values instead.
  df['grad_semestre'] = df['grad_semestre'].replace(['None','0'])


array([None, '2', '1'], dtype=object)

In [7]:
df['grad_annio'] = df['grad_annio'].fillna('0').astype(str)
df['grad_semestre'] = df['grad_semestre'].fillna('0').astype(str)
df['Fecha_grado'] = df['grad_annio'] + '-' + df['grad_semestre']

#df['Fecha_grado'] = df['grad_annio'].astype(str) + '-' + df['grad_semestre'].astype(str)

In [8]:
df = df[['Identificacion', 'Nombre_Estudiante', 'Periodo', 'Cod_Unidad',
       'Status',  'ciclo', 'AÑO_MEN',  'grad_annio', 'grad_semestre', 'num_repeticiones',
       'Primer_curso', 'ultimo_curso', 'Fecha_grado']]

In [9]:
df = df[df['ultimo_curso']== 'SI']

In [10]:
df['Fecha_grado']= df['Fecha_grado'].fillna(0)
df['Fecha_grado'].unique()

array(['0-0', '2025-2', '2022-2', '2023-2', '2024-1', '2026-1', '2023-1',
       '2020-1', '2021-2', '2024-2', '2022-1', '2025-1', '2019-2',
       '2019-1', '2013-2', '2013-1', '2015-2', '2016-1', '2021-1',
       '2020-2', '2018-2', '2016-2', '2017-1', '2018-1', '2017-2',
       '2014-2', '2015-1', '2014-1', '2012-2', '2008-1', '2009-2',
       '2009-1', '2008-2', '2012-1', '2006-1', '2011-2', '2010-2',
       '2011-1', '2010-1', '2006-2', '1999-1', '2007-2', '2007-1',
       '2005-1', '2004-2', '2005-2', '2004-1', '2003-2', '2003-1',
       '2001-1', '1997-1', '1991-2', '2002-1', '2000-1', '1998-2',
       '2002-2', '1997-2', '1995-2', '1999-2', '1994-2', '1996-2',
       '2001-2'], dtype=object)

In [11]:
import pandas as pd
import numpy as np

# Asumiendo que df es tu DataFrame y tiene las columnas 'Fecha_inicio' y 'Fecha_fin' en el formato "año-semestre".
# Extraer año y semestre de 'Fecha_inicio'
df[['Año_grado', 'Semestre_grado']] = df['Fecha_grado'].str.split('-', expand=True).fillna(0).astype(int)

# Extraer año y semestre de 'Fecha_fin'
df[['Año_fin', 'Semestre_fin']] = df['AÑO_MEN'].str.split('-', expand=True).fillna(0).astype(int)

# Calcular la diferencia en semestres
df['Diferencia_Semestres'] = ((df['Año_grado'] - df['Año_fin']) * 2) + (df['Semestre_grado'] - df['Semestre_fin'])

df.loc[df['Diferencia_Semestres'] < 0, 'Diferencia_Semestres'] = np.nan

In [12]:
df['Tuvo_grado_oportuno'] = np.where(
    pd.notna(df['Diferencia_Semestres']) & (df['Diferencia_Semestres'] > 2),
    'No_tuvo_grado_oportuno',
    np.where(
        pd.notna(df['Diferencia_Semestres']),
        'Fue grado oportuno',
        None  # Aquí usamos None en vez de np.nan
    )
).astype(object)  # Forzamos a que todo sea de tipo object


In [13]:
# import numpy as np


# df['Tuvo_grado_oportuno'] = np.where(  df['Diferencia_Semestres'] >= 2, 
#                                      'No_tuvo_grado_oportuno',  # Si la condición es verdadera
#                                      'Fue grado oportuno')  

In [14]:
df= df[['Identificacion', 'Nombre_Estudiante', 'Periodo', 'ciclo', 'AÑO_MEN', 'ultimo_curso', 'Fecha_grado', 'Diferencia_Semestres', 'Tuvo_grado_oportuno']]
df[df['Identificacion']== '1023922245']

#df.to_csv('Grado_Oportuno_historico_cruce_F.csv', index=False)

,Identificacion,Nombre_Estudiante,Periodo,ciclo,AÑO_MEN,ultimo_curso,Fecha_grado,Diferencia_Semestres,Tuvo_grado_oportuno
381448,1023922245,SANCHEZ ROMERO CRISTHIAN ALIRIO,2018A,TECNICO,2018-1,SI,2023-2,11.0,No_tuvo_grado_oportuno
381455,1023922245,SANCHEZ ROMERO CRISTHIAN ALIRIO,2022B,TECNOLOGO,2022-2,SI,2023-2,2.0,Fue grado oportuno
381456,1023922245,SANCHEZ ROMERO CRISTHIAN ALIRIO,2023A,PROFESIONAL,2023-1,SI,2023-2,1.0,Fue grado oportuno


In [15]:
df

,Identificacion,Nombre_Estudiante,Periodo,ciclo,AÑO_MEN,ultimo_curso,Fecha_grado,Diferencia_Semestres,Tuvo_grado_oportuno
0,00010105021,FULA FAJARDO SEBASTIAN ORLANDO,2017B,TECNICO,2017-2,SI,0-0,NaN,None
2,00010308259,GUARNIZO OTAVO VALERIA,2018A,TECNICO,2018-1,SI,0-0,NaN,None
3,00010505011,BARRIOS ORTEGA YERYS SOFIA,2017B,TECNICO,2017-2,SI,0-0,NaN,None
4,00010605938,GOMEZ CLAVIJO ANGIE LIZETH,2017B,TECNICO,2017-2,SI,0-0,NaN,None
5,00010606926,MORENO SALINAS FREDDY SNEYDER,2017A,TECNICO,2017-1,SI,0-0,NaN,None
...,...,...,...,...,...,...,...,...,...
1092301,9994800,RAMIREZ PARADA CESAR AUGUSTO,2016A,PROFESIONAL,2016-1,SI,2016-2,1.0,Fue grado oportuno
1092304,999604,ALVIREZ HUIZA ANTHONIN ZAQUIEL,26P04,TECNICO,2026-2,SI,0-0,NaN,None
1092312,None,Sin Nombre,2019A,PROFESIONAL,2019-1,SI,0-0,NaN,None
1092330,None,Sin Nombre,23V05,TECNICO,2023-2,SI,0-0,NaN,None


In [16]:
df['Identificacion'].nunique()

287342

In [17]:
# # Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'Grado_Oportuno_historico'
schema_name = 'DBARON'
# Send the data to sql server
df.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()

##Aqui se valida de nuevo

In [18]:

from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
#from datetime import datetime
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)

with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
       select distinct a.*, b.AÑO_MEN, b.Tuvo_grado_oportuno
            from Grados.Consolidado_Grados a
            left join DBARON.Grado_Oportuno_historico  b
            on a.NUMERO = b.Identificacion
            and a.NIVEL = b.ciclo
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

Conexion a la base de datos realizada


In [19]:
df['NIVEL'].unique()
df

,Nombres,genero_code,tipo_doc_unico,NUMERO,grad_annio,grad_semestre,PLAN,snies,METODOLOGIA,NIVEL,...,opcion_grado,lat,lon,departamento,ciudad,localidad_normalizada,CEREMONIA,TIPO_CONVOCATORIA,AÑO_MEN,Tuvo_grado_oportuno
0,TOBAR MORAN WILSON ARMANDO,o,CC,1085922428,2024,2,1110,108558,VIRTUAL,TECNICO,...,None,0.823253,-77.635337,NARIÑO,IPIALES,Sin localidad,None,None,2022-2,No_tuvo_grado_oportuno
1,FABIAN ANDRES LOPEZ GIL,o,CC,1023963637,2024,1,VG001,104396,VIRTUAL,PROFESIONAL,...,HOMOLOGACION DIPLOMADO,4.646923,-74.156261,DISTRITO CAPITAL,BOGOTA,Kennedy,None,None,2023-1,Fue grado oportuno
2,AARON DE JESUS BUZON HERRERA,o,CC,72268429,2022,2,1030,6101,PRESENCIAL,TECNICO,...,Homologación,11.233151,-74.207805,MAGDALENA,SANTA MARTA,Histórica – Rodrigo de Bastidas,None,None,2019-1,No_tuvo_grado_oportuno
3,AARON SANTIAGO SUAREZ ABELLO,o,CC,1007365381,2022,2,1036,52606,PRESENCIAL,TECNICO,...,None,4.517678,-74.118932,DISTRITO CAPITAL,BOGOTA,Usme,None,None,2021-1,No_tuvo_grado_oportuno
4,AARON SANTIAGO SUAREZ ABELLO,o,CC,1007365381,2025,2,ESPMA,108460,VIRTUAL,ESPECIALIZACION,...,None,4.516044,-74.118056,DISTRITO CAPITAL,BOGOTA,Usme,POR SECRETARIA,2025-4,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186676,ZURLEY TAPASCO RIOS,a,CC,40778548,2007,2,1013,2380,DISTANCIA,TECNICO,...,None,NaN,NaN,None,None,None,None,None,None,None
186677,ZURY MISAIDA RUIZ MERCADO,a,CC,1102856365,2015,1,6013,54159,PRESENCIAL,TECNICO,...,CURSO NORMAL,9.306432,-75.396993,SUCRE,SINCELEJO,Sin localidad,None,None,2014-1,None
186678,ZURY NATALIA OSORIO CASTAÑO,a,CC,1110542105,2017,1,2022,54235,PRESENCIAL,TECNICO,...,CURSO NORMAL,4.440000,-75.186632,TOLIMA,IBAGUE,Comuna 5 - Jordán,None,None,None,None
186679,ZURY NATALIA OSORIO CASTAÑO,a,CC,1110542105,2020,1,2022,54215,PRESENCIAL,PROFESIONAL,...,CURSO NORMAL,4.440000,-75.186632,TOLIMA,IBAGUE,Comuna 5 - Jordán,None,None,2019-2,Fue grado oportuno


In [20]:
df.loc[df['Tuvo_grado_oportuno'].isna() & (df['NIVEL'] != 'ESPECIALIZACION'), 'Tuvo_grado_oportuno'] = 'No_tuvo_grado_oportuno'

In [21]:
#df.loc[df['Tuvo_grado_oportuno'].isna() , 'Tuvo_grado_oportuno'] = 'Especializacion'

In [22]:
df['Tuvo_grado_oportuno'].unique()
df[df['Tuvo_grado_oportuno'].isna()]
#df[df['Tuvo_grado_oportuno'].isnull()]
df

,Nombres,genero_code,tipo_doc_unico,NUMERO,grad_annio,grad_semestre,PLAN,snies,METODOLOGIA,NIVEL,...,opcion_grado,lat,lon,departamento,ciudad,localidad_normalizada,CEREMONIA,TIPO_CONVOCATORIA,AÑO_MEN,Tuvo_grado_oportuno
0,TOBAR MORAN WILSON ARMANDO,o,CC,1085922428,2024,2,1110,108558,VIRTUAL,TECNICO,...,None,0.823253,-77.635337,NARIÑO,IPIALES,Sin localidad,None,None,2022-2,No_tuvo_grado_oportuno
1,FABIAN ANDRES LOPEZ GIL,o,CC,1023963637,2024,1,VG001,104396,VIRTUAL,PROFESIONAL,...,HOMOLOGACION DIPLOMADO,4.646923,-74.156261,DISTRITO CAPITAL,BOGOTA,Kennedy,None,None,2023-1,Fue grado oportuno
2,AARON DE JESUS BUZON HERRERA,o,CC,72268429,2022,2,1030,6101,PRESENCIAL,TECNICO,...,Homologación,11.233151,-74.207805,MAGDALENA,SANTA MARTA,Histórica – Rodrigo de Bastidas,None,None,2019-1,No_tuvo_grado_oportuno
3,AARON SANTIAGO SUAREZ ABELLO,o,CC,1007365381,2022,2,1036,52606,PRESENCIAL,TECNICO,...,None,4.517678,-74.118932,DISTRITO CAPITAL,BOGOTA,Usme,None,None,2021-1,No_tuvo_grado_oportuno
4,AARON SANTIAGO SUAREZ ABELLO,o,CC,1007365381,2025,2,ESPMA,108460,VIRTUAL,ESPECIALIZACION,...,None,4.516044,-74.118056,DISTRITO CAPITAL,BOGOTA,Usme,POR SECRETARIA,2025-4,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186676,ZURLEY TAPASCO RIOS,a,CC,40778548,2007,2,1013,2380,DISTANCIA,TECNICO,...,None,NaN,NaN,None,None,None,None,None,None,No_tuvo_grado_oportuno
186677,ZURY MISAIDA RUIZ MERCADO,a,CC,1102856365,2015,1,6013,54159,PRESENCIAL,TECNICO,...,CURSO NORMAL,9.306432,-75.396993,SUCRE,SINCELEJO,Sin localidad,None,None,2014-1,No_tuvo_grado_oportuno
186678,ZURY NATALIA OSORIO CASTAÑO,a,CC,1110542105,2017,1,2022,54235,PRESENCIAL,TECNICO,...,CURSO NORMAL,4.440000,-75.186632,TOLIMA,IBAGUE,Comuna 5 - Jordán,None,None,None,No_tuvo_grado_oportuno
186679,ZURY NATALIA OSORIO CASTAÑO,a,CC,1110542105,2020,1,2022,54215,PRESENCIAL,PROFESIONAL,...,CURSO NORMAL,4.440000,-75.186632,TOLIMA,IBAGUE,Comuna 5 - Jordán,None,None,2019-2,Fue grado oportuno


In [23]:
df_filtrado = df[df['Tuvo_grado_oportuno'].notna()]

In [24]:
df_filtrado = df.loc[df['Tuvo_grado_oportuno'].notna()]
#df_filtrado.to_csv('Grado_Oportuno_Historico_1.csv', index=False)
df_filtrado['grad_semestre'] = df_filtrado['grad_semestre'].str.replace('\.0$', '', regex=True)


<>:3: SyntaxWarning: invalid escape sequence '\.'
<>:3: SyntaxWarning: invalid escape sequence '\.'
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_12628\4058290774.py:3: SyntaxWarning: invalid escape sequence '\.'
  df_filtrado['grad_semestre'] = df_filtrado['grad_semestre'].str.replace('\.0$', '', regex=True)
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_12628\4058290774.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado['grad_semestre'] = df_filtrado['grad_semestre'].str.replace('\.0$', '', regex=True)


In [25]:
df_filtrado['grad_annio'] = df_filtrado['grad_annio'].fillna('0').astype(str)
df_filtrado['grad_semestre'] = df_filtrado['grad_semestre'].fillna('0').astype(str)
df_filtrado['Fecha_grado'] = df_filtrado['grad_annio'] + '-' + df_filtrado['grad_semestre']

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_12628\4124796657.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado['grad_annio'] = df_filtrado['grad_annio'].fillna('0').astype(str)
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_12628\4124796657.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado['grad_semestre'] = df_filtrado['grad_semestre'].fillna('0').astype(str)
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_12628\4124796657.py:3: SettingWithCopy

In [26]:
df_filtrado['AÑO_MEN'].unique()
#df_filtrado['AÑO_MEN'] = df_filtrado['AÑO_MEN'].fillna('0-0')

array(['2022-2', '2023-1', '2019-1', '2021-1', '2013-1', None, '2020-1',
       '2022-1', '2024-1', '2015-1', '2016-1', '2023-2', '2024-2',
       '2018-1', '2020-2', '2014-1', '2018-2', '2017-1', '2015-2',
       '2017-2', '2019-2', '2025-2', '2014-2', '2016-2', '2013-2',
       '2025-1', '2021-2', '2012-1', '2012-2', '2026-1', '2026-2',
       '2011-2'], dtype=object)

In [27]:
# import pandas as pd
# import numpy as np

# # Asumiendo que df es tu DataFrame y tiene las columnas 'Fecha_inicio' y 'Fecha_fin' en el formato "año-semestre".
# # Extraer año y semestre de 'Fecha_inicio'
# df_filtrado[['Año_grado', 'Semestre_grado']] = df_filtrado['Fecha_grado'].str.split('-', expand=True).astype(int)

# # Extraer año y semestre de 'Fecha_fin'
# df_filtrado[['Año_fin', 'Semestre_fin']] = df_filtrado['AÑO_MEN'].str.split('-', expand=True).astype(int)

# # Calcular la diferencia en semestres
# df_filtrado['Diferencia_Semestres'] = ((df_filtrado['Año_grado'] - df_filtrado['Año_fin']) * 2) + (df_filtrado['Semestre_grado'] - df_filtrado['Semestre_fin'])

# df_filtrado.loc[df_filtrado['Diferencia_Semestres'] < 0, 'Diferencia_Semestres'] = np.nan


In [28]:
import pandas as pd
import numpy as np

# Asumiendo que df_filtrado es tu DataFrame
# Extraer año y semestre de 'Fecha_grado'
df_filtrado[['Año_grado', 'Semestre_grado']] = df_filtrado['Fecha_grado'].str.split('-', expand=True).apply(pd.to_numeric, errors='coerce')

# Extraer año y semestre de 'AÑO_MEN'
df_filtrado[['Año_fin', 'Semestre_fin']] = df_filtrado['AÑO_MEN'].str.split('-', expand=True).apply(pd.to_numeric, errors='coerce')

# Calcular la diferencia en semestres
df_filtrado['Diferencia_Semestres'] = ((df_filtrado['Año_grado'] - df_filtrado['Año_fin']) * 2) + (df_filtrado['Semestre_grado'] - df_filtrado['Semestre_fin'])

# Asignar NaN a diferencias negativas
df_filtrado.loc[df_filtrado['Diferencia_Semestres'] < 0, 'Diferencia_Semestres'] = np.nan


C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_12628\4017461670.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado[['Año_grado', 'Semestre_grado']] = df_filtrado['Fecha_grado'].str.split('-', expand=True).apply(pd.to_numeric, errors='coerce')
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_12628\4017461670.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado[['Año_grado', 'Semestre_grado']] = df_filtrado['Fecha_grado'].str.split('-', expand=True).apply(pd.to_nume

In [29]:
df_filtrado['Tuvo_grado_oportuno'] = np.where(
    pd.notna(df_filtrado['Diferencia_Semestres']) & (df_filtrado['Diferencia_Semestres'] > 2),
    'No_tuvo_grado_oportuno',
    np.where(
        pd.notna(df_filtrado['Diferencia_Semestres']),
        'Fue grado oportuno',
        None  # <- Usamos None en vez de np.nan para evitar conflictos de tipo
    )
).astype(object)  # <- Esto garantiza que todo pueda convivir como tipo object


C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_12628\691709718.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado['Tuvo_grado_oportuno'] = np.where(


In [30]:
df_filtrado

,Nombres,genero_code,tipo_doc_unico,NUMERO,grad_annio,grad_semestre,PLAN,snies,METODOLOGIA,NIVEL,...,CEREMONIA,TIPO_CONVOCATORIA,AÑO_MEN,Tuvo_grado_oportuno,Fecha_grado,Año_grado,Semestre_grado,Año_fin,Semestre_fin,Diferencia_Semestres
0,TOBAR MORAN WILSON ARMANDO,o,CC,1085922428,2024,2,1110,108558,VIRTUAL,TECNICO,...,None,None,2022-2,No_tuvo_grado_oportuno,2024-2,2024,2.0,2022.0,2.0,4.0
1,FABIAN ANDRES LOPEZ GIL,o,CC,1023963637,2024,1,VG001,104396,VIRTUAL,PROFESIONAL,...,None,None,2023-1,Fue grado oportuno,2024-1,2024,1.0,2023.0,1.0,2.0
2,AARON DE JESUS BUZON HERRERA,o,CC,72268429,2022,2,1030,6101,PRESENCIAL,TECNICO,...,None,None,2019-1,No_tuvo_grado_oportuno,2022-2,2022,2.0,2019.0,1.0,7.0
3,AARON SANTIAGO SUAREZ ABELLO,o,CC,1007365381,2022,2,1036,52606,PRESENCIAL,TECNICO,...,None,None,2021-1,No_tuvo_grado_oportuno,2022-2,2022,2.0,2021.0,1.0,3.0
5,ABDERSON ALFONSO PACHON TORRES,o,CC,79319704,2015,2,1071,53142,DISTANCIA,TECNOLOGO,...,None,None,2013-1,No_tuvo_grado_oportuno,2015-2,2015,2.0,2013.0,1.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186676,ZURLEY TAPASCO RIOS,a,CC,40778548,2007,2,1013,2380,DISTANCIA,TECNICO,...,None,None,None,None,2007-2,2007,2.0,NaN,NaN,NaN
186677,ZURY MISAIDA RUIZ MERCADO,a,CC,1102856365,2015,1,6013,54159,PRESENCIAL,TECNICO,...,None,None,2014-1,Fue grado oportuno,2015-1,2015,1.0,2014.0,1.0,2.0
186678,ZURY NATALIA OSORIO CASTAÑO,a,CC,1110542105,2017,1,2022,54235,PRESENCIAL,TECNICO,...,None,None,None,None,2017-1,2017,1.0,NaN,NaN,NaN
186679,ZURY NATALIA OSORIO CASTAÑO,a,CC,1110542105,2020,1,2022,54215,PRESENCIAL,PROFESIONAL,...,None,None,2019-2,Fue grado oportuno,2020-1,2020,1.0,2019.0,2.0,1.0


In [31]:
df_filtrado.loc[df_filtrado['Tuvo_grado_oportuno']== 'nan']
#df_filtrado

,Nombres,genero_code,tipo_doc_unico,NUMERO,grad_annio,grad_semestre,PLAN,snies,METODOLOGIA,NIVEL,...,CEREMONIA,TIPO_CONVOCATORIA,AÑO_MEN,Tuvo_grado_oportuno,Fecha_grado,Año_grado,Semestre_grado,Año_fin,Semestre_fin,Diferencia_Semestres


In [32]:
df_filtrado.loc[df_filtrado['Tuvo_grado_oportuno']== 'nan', 'Tuvo_grado_oportuno'] = 'No_tuvo_grado_oportuno'

s = df_filtrado[df_filtrado['grad_annio']=='2024']
s

,Nombres,genero_code,tipo_doc_unico,NUMERO,grad_annio,grad_semestre,PLAN,snies,METODOLOGIA,NIVEL,...,CEREMONIA,TIPO_CONVOCATORIA,AÑO_MEN,Tuvo_grado_oportuno,Fecha_grado,Año_grado,Semestre_grado,Año_fin,Semestre_fin,Diferencia_Semestres
0,TOBAR MORAN WILSON ARMANDO,o,CC,1085922428,2024,2,1110,108558,VIRTUAL,TECNICO,...,None,None,2022-2,No_tuvo_grado_oportuno,2024-2,2024,2.0,2022.0,2.0,4.0
1,FABIAN ANDRES LOPEZ GIL,o,CC,1023963637,2024,1,VG001,104396,VIRTUAL,PROFESIONAL,...,None,None,2023-1,Fue grado oportuno,2024-1,2024,1.0,2023.0,1.0,2.0
16,ABADIA TABORDA CARLOS ARMANDO,o,CC,94312992,2024,2,VG001,104396,VIRTUAL,PROFESIONAL,...,None,None,2023-2,Fue grado oportuno,2024-2,2024,2.0,2023.0,2.0,2.0
17,ABDALA ISAAC SAAD DIAZ,o,CC,8569083,2024,1,VTG12,105087,VIRTUAL,TECNOLOGO,...,None,None,2023-1,Fue grado oportuno,2024-1,2024,1.0,2023.0,1.0,2.0
47,ABELLO DOMÍNGUEZ BREINER DE JESÚS,o,CC,1152942566,2024,2,VG002,104397,VIRTUAL,TECNICO,...,None,None,2023-2,Fue grado oportuno,2024-2,2024,2.0,2023.0,2.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186655,ZULUAGA RESTREPO JOHAN SEBASTIAN,o,CC,1112779009,2024,2,3029,104862,VIRTUAL,PROFESIONAL,...,None,None,2024-2,Fue grado oportuno,2024-2,2024,2.0,2024.0,2.0,0.0
186668,ZUÑIGA ARRIETA YONATHAN ALEXANDER,o,CC,1001276672,2024,2,VG001,104396,VIRTUAL,PROFESIONAL,...,None,None,2024-2,Fue grado oportuno,2024-2,2024,2.0,2024.0,2.0,0.0
186669,ZUÑIGA BETANCOURT MARIA ANGELICA,a,CC,1003459140,2024,2,1073,108543,VIRTUAL,TECNOLOGO,...,None,None,2023-2,Fue grado oportuno,2024-2,2024,2.0,2023.0,2.0,2.0
186671,ZUÑIGA VILLACORTE LUIS CARLOS,o,CC,87218806,2024,2,VG001,104397,VIRTUAL,TECNICO,...,None,None,2023-1,No_tuvo_grado_oportuno,2024-2,2024,2.0,2023.0,1.0,3.0


In [33]:
df_filtrado = df_filtrado[['Nombres', 'genero_code', 'tipo_doc_unico', 'NUMERO', 'grad_annio',
       'grad_semestre', 'PLAN', 'snies', 'METODOLOGIA', 'NIVEL', 'TITULACION',
       'ESCUELA', 'numero_de_credencial',
       'codigo_municipio_grado', 'SEDE_GRADO', 'acta', 'No_de_Registro',
       'folio', 'libro', 'ciclo_grado', 'Celular_Sugerido',
       'Telefono_Sugerido', 'Otros_Sugeridos', 'Correo_Personal_Sugerido',
       'PROGRAMA_PROFESIONAL', 'NOM_PROGRAMA_PROFESIONAL', 'SEDE_GRADO_2',
       'SCORE_M0', 'opcion_grado', 'AÑO_MEN', 'Tuvo_grado_oportuno',
       'Fecha_grado', 'Año_grado', 'Semestre_grado', 'Año_fin', 'Semestre_fin',
       'Diferencia_Semestres']]

In [34]:
#df_filtrado.to_csv('Grado_Oportuno_Historico_3.csv', index=False)

In [35]:
df_filtrado[df_filtrado['NUMERO']=='1022418911']

,Nombres,genero_code,tipo_doc_unico,NUMERO,grad_annio,grad_semestre,PLAN,snies,METODOLOGIA,NIVEL,...,SCORE_M0,opcion_grado,AÑO_MEN,Tuvo_grado_oportuno,Fecha_grado,Año_grado,Semestre_grado,Año_fin,Semestre_fin,Diferencia_Semestres
141903,Sandra Carolina Laguna Flechas,a,CC,1022418911,2023,2,VG001,104396,VIRTUAL,PROFESIONAL,...,No Contactado,Homologación,2024-1,None,2023-2,2023,2.0,2024.0,1.0,NaN
180104,SANDRA CAROLINA LAGUNA FLECHAS,a,CC,1022418911,2023,None,VG001,104396,VIRTUAL,PROFESIONAL,...,No Contactado,Homologación,2024-1,None,2023-None,2023,NaN,2024.0,1.0,NaN


In [36]:
df_filtrado.shape

(183109, 37)

In [37]:
# # Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'Grado_Oportuno_historico'
schema_name = 'SRUBIO'
# Send the data to sql server
df_filtrado.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()